# Searching 20-newsgroups 

In [19]:
import thematic_search as ts
import numpy as np
import pandas as pd

newsgroups_df = pd.read_parquet("hf://datasets/lmcinnes/20newsgroups_embedded/data/train-00000-of-00001.parquet")
newsgroups_df.head(1)

,post,newsgroup,embedding,map
0,\n\nI am sure some bashers of Pens fans are pr...,rec.sport.hockey,"[-0.04380008950829506, 0.08495834469795227, -0...","[-0.13199903070926666, 10.1972017288208]"


In [ ]:
tags = np.unique(newsgroups_df['newsgroup'].to_numpy())

from collections import defaultdict
def build_tree(paths):
    tree = defaultdict(set)
    tree["root"]
    for p in paths:
        parts = p.split(".")
        for i in range(len(parts)):
            node = ".".join(parts[:i+1])
            parent = "root" if i == 0 else ".".join(parts[:i])
            tree[parent].add(node)
            tree[node]
    return {k: sorted(v) for k, v in tree.items()}

tree = build_tree(tags)
cluster_tree, cluster_labels = ts.utils.convert_string_tree(tree) 
ts.utils.print_tree(cluster_tree, cluster_labels=cluster_labels)

root
--alt
----alt.atheism
--comp
----comp.graphics
----comp.os
------comp.os.ms-windows
--------comp.os.ms-windows.misc
----comp.sys
------comp.sys.ibm
--------comp.sys.ibm.pc
----------comp.sys.ibm.pc.hardware
------comp.sys.mac
--------comp.sys.mac.hardware
----comp.windows
------comp.windows.x
--misc
----misc.forsale
--rec
----rec.autos
----rec.motorcycles
----rec.sport
------rec.sport.baseball
------rec.sport.hockey
--sci
----sci.crypt
----sci.electronics
----sci.med
----sci.space
--soc
----soc.religion
------soc.religion.christian
--talk
----talk.politics
------talk.politics.guns
------talk.politics.mideast
------talk.politics.misc
----talk.religion
------talk.religion.misc


In [21]:
tag_to_tuple = {v:k for k,v in cluster_labels.items()}

data = []
for tag in tree.keys():
    layer, cluster = tag_to_tuple[tag]
    data.append({
        'uid':ts.utils.topic_uid((layer,cluster)),
        'name':tag,
        'layer':layer,
        'cluster':cluster,
    })

topic_df = pd.DataFrame(data)
topic_df.head(2)

,uid,name,layer,cluster
0,ABQB,root,5,0
1,AAQB,alt,1,0


In [22]:

cluster_matrices = []
n_layers = max(topic_df['layer'].values) # Don't count the root
n_samples = len(newsgroups_df)
for l in range(n_layers):
    tags_in_layer = topic_df[topic_df['layer']==l]['cluster'].to_list()
    n_tags = len(tags_in_layer)
    matrix = np.zeros((n_samples, n_tags))
    for tag in tags_in_layer:
        tag_name = cluster_labels[(l, tag)]
        matrix[:, tag] =  newsgroups_df['newsgroup'].apply(
            lambda x: x == tag_name
        )
    cluster_matrices.append(matrix)

for matrix in cluster_matrices:
    print(matrix.shape)


(18170, 20)
(18170, 11)
(18170, 5)
(18170, 1)
(18170, 1)


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

soft_cluster_tree = ts.SoftClusterTree(
    cluster_matrices,
    cluster_tree,
)
topic_db = ts.TopicDatabase(
    soft_cluster_tree = soft_cluster_tree,
    embedding_vectors = np.stack(newsgroups_df['embedding'].values),
    reduced_vectors = np.stack(newsgroups_df['map'].values),
    document_df = newsgroups_df[['post', 'newsgroup']],
    topic_df = topic_df,
    embedding_model = model,
)
topic_db

In [25]:
topic_df.head(5)

,uid,name,layer,cluster
0,ABQB,root,5,0
1,AAQB,alt,1,0
2,AAAB,alt.atheism,0,0
3,ABAB,comp,4,0
4,AAAC,comp.graphics,0,1


In [26]:
topic_db.q.topic_uid('AAAC').inside().documents()

,post,newsgroup
22,"I don't have nor Imagine nor Real 3d, but as o...",comp.graphics
27,"Using the VMODE command, all you need to do is...",comp.graphics
41,I am looking for a package which takes as inpu...,comp.graphics
62,I am currently using POVRay on Mac and was won...,comp.graphics
63,Here now some initial references; best regards...,comp.graphics
...,...,...
18081,Has anyone seen hallusions? You can buy a pos...,comp.graphics
18112,\n: Can anyone tell me where to find a MPEG vi...,comp.graphics
18117,\n For a commerical package try WAVE from Pr...,comp.graphics
18124,"Hi everybody,\n\nCan anyone name an anonymous ...",comp.graphics
